<a href="https://colab.research.google.com/github/Aeagon07/Genarative-AI/blob/main/Tools_In_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## Built-In DuckDuckGo Search

In [3]:
!pip install -U ddgs
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

res = search_tool.invoke('IPL News')

print(res)

IPL2026: All rules and playing regulations detailed A complete breakdown of all the latest rules, match regulations, and key playing conditions in Indian Premier League 2026, includingnew... Latest and breaking CricketNews, match peviews, reviews, player interviews of theIPL| Indian Premier League 2026 on Cricbuzz.com IPL2026 :Explore the latest updates, scores, and highlights of theIPL2026 on News18 Cricket. Stay informed about every match fromIPL2026 at news18.com Catch all the latest & breakingnewsonIPL2026 Cricket series. Get all updates onIPL2026 Cricket team players and coaches at NDTV Sports. IPLnewsand injury updates on Ishan Kishan, Pat Cummins, Matheesha Pathirana, Gudakesh Motie & more ahead of 2026 season.


In [12]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


## Built-In - Shell Tool

In [4]:
from langchain_community.tools import ShellTool

# With this tools you can execute command on command line
shell_tool = ShellTool()

res = shell_tool.invoke('ls')

print(res)

Executing command:
 ls
sample_data



/usr/local/lib/python3.12/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


# Custom Tools

In [5]:
from langchain_core.tools import tool

In [6]:
# Step 1 => Create Function

def multiply(a, b):
  """Multiply two Numbers"""
  return a * b

In [7]:
#Step 2 => Add type hints
def multiply(a: int, b: int) -> int:
  """Multiply two numbers"""
  return a * b

In [8]:
# Step 3 => Add tool Decorator

@tool
def multiply(a: int, b: int) -> int:
  """Multiply two numbers"""
  return a * b

In [9]:
res = multiply.invoke({"a": 3, "b": 5})

In [10]:
print(res)

15


In [11]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [13]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


# Method - 2 : Using Structured Tool

In [16]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [17]:
class MultiplyInput(BaseModel):
  a: int = Field(required=True, description="The first number to add")
  b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_443/1787733605.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_443/1787733605.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The first number to add")


In [18]:
def multiply_func(a: int, b: int) -> int:
  return a * b

In [20]:
multiply_tool = StructuredTool.from_function(
    func = multiply_func,
    name = "multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [21]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The first number to add', 'title': 'B', 'type': 'integer'}}


# Method - 3 : Using BaseTool Class

In [22]:
from langchain.tools import BaseTool
from typing import Type

In [23]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(required=True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_443/908171234.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required=True, description="The first number to add")
/tmp/ipykernel_443/908171234.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


In [25]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput # Cause this is Pydantic schema that we define earliar

    def _run(self, a: int, b: int) -> int:
        return a * b

  # This is inheriting the BaseTool Class so, This is another custom tool in langchain

In [26]:
multiply_tool = MultiplyTool()

In [27]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


# ToolKit

In [28]:
from langchain_core.tools import tool

# Custom tool
@tool
def add(a: int, b: int) -> int:
  """Add two numbers"""
  return a + b

@tool
def multiply(a: int, b:int) -> int:
  """Multiplt two numbers"""
  return a * b

In [29]:
class MathToolKit:
  def get_tools(self):
    return [add, multiply]

In [30]:
toolkit = MathToolKit()

tools = toolkit.get_tools()

for tool in tools:
  print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiplt two numbers
